In [1]:
!pip install pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLake").getOrCreate()

In [2]:
import os, shutil
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)
shutil.copy("/sales.csv", "data_lake/bronze/sales.csv")
print(os.listdir("data_lake/bronze"))

['sales.csv']


In [4]:
os.listdir("datalake/bronze")


['sales_raw.csv']

In [3]:
from pyspark.sql.functions import (col, trim, initcap, to_date, coalesce, when, lit,
                                   sum as _sum)

df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()
rows_before = df.count()
print("Total records:", rows_before)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|
|  100004|      C0051|

In [4]:
# Null values
df.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).show()

# Exact duplicate rows
print("Exact duplicate rows:", df.count() - df.dropDuplicates().count())

# Invalid numbers
df.filter((col("quantity") <= 0) | (col("unit_price") <= 0) |
          (col("discount_percent") < 0) | (col("discount_percent") > 100)).show(10)

# Text inconsistency
df.groupBy("category").count().show()

# Dates not in yyyy-MM-dd format
df.filter(~col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")).select("order_date").show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Exact duplicate rows: 10
+--------+-----------+---------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|  customer_name|             product|       category|quantity|unit_

In [5]:
spark.conf.set("spark.sql.ansi.enabled", "false")  # strict mode off, so bad dates become null instead of an error

silver = df

# 1. Trim spaces and standardise capitalisation
for c in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    silver = silver.withColumn(c, initcap(trim(col(c))))

# 2. Remove exact duplicate rows only
silver = silver.dropDuplicates()

# 3. Fill missing text values
silver = silver.fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

# 4. Invalid numbers -> empty -> filled with median
silver = silver.withColumn("quantity", when(col("quantity") <= 0, lit(None)).otherwise(col("quantity")))
silver = silver.withColumn("unit_price", when(col("unit_price") <= 0, lit(None)).otherwise(col("unit_price")))
silver = silver.withColumn("discount_percent",
    when((col("discount_percent") < 0) | (col("discount_percent") > 100), lit(None))
    .otherwise(col("discount_percent")))

medians = {c: int(silver.approxQuantile(c, [0.5], 0.01)[0])
           for c in ["quantity", "unit_price", "discount_percent"]}
print("Median values used:", medians)
silver = silver.fillna(medians)

# 5. One consistent date format; impossible dates (like 31/02/2026) stay empty
silver = silver.withColumn("order_date", coalesce(
    to_date(col("order_date"), "yyyy-MM-dd"),
    to_date(col("order_date"), "dd/MM/yyyy")))

rows_after = silver.count()
print("Rows before preprocessing:", rows_before)
print("Rows after preprocessing:", rows_after)

Median values used: {'quantity': 5, 'unit_price': 1810, 'discount_percent': 20}
Rows before preprocessing: 1000
Rows after preprocessing: 990


In [6]:
silver.write.mode("overwrite").option("header", True).csv("data_lake/silver/preprocessed_sales")

os.makedirs("silver", exist_ok=True)
silver.toPandas().to_csv("silver/sales_silver.csv", index=False)
print("Silver layer saved")

Silver layer saved


In [7]:
check = spark.read.csv("data_lake/silver/preprocessed_sales", header=True, inferSchema=True)
check.show(5)
check.printSchema()
print("Rows before:", rows_before, "| Rows in Silver:", check.count())

+--------+-----------+------------------+----------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|         product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+----------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100008|      C0124|   Tanvi Chaudhary|    Dining Table|Furniture|       6|     25970|              40|Cash On Delivery|     Delhi|      Delhi|2025-09-25|   Delivered|
|  100123|      C0272|      Sanjay Naidu|       Atta 10kg|  Grocery|       4|       560|               0|             Upi|Coimbatore| Tamil Nadu|2026-01-15|     Shipped|
|  100331|      C0005|      Sakshi Joshi|        Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-

## Preprocessing operations performed
- Trimmed extra spaces and standardised capitalisation in text columns
- Removed exact duplicate rows
- Filled missing customer_name, payment_method and city with "Unknown"
- Replaced invalid quantity, unit_price and discount_percent values with the column median
- Converted order_date to one date format; impossible dates left empty